# GCM BA - 7c. Ad hoc test: spatial coherence and areal extremes of MBCn

**Question (raised by the supervisor for any pixel-wise correction).** A univariate quantile correction per pixel is
monotonic, so it keeps the order of the days at every pixel: the rank (Spearman) correlation between pixels and the
rank autocorrelation are those of the model by construction; only the Pearson statistics can change. MBCn is applied
per pixel too, but its last step reorders the univariate result of each pixel along the ranks of its own multivariate
transform: the order of the days can change at each pixel independently of its neighbours, so the spatial coherence
and the persistence can weaken. 7b showed a small loss of persistence (lag-1 autocorrelation of the winter tasmax
anomaly 0.797 F1 → 0.778 MBCn, ERA5 0.810). What about the spatial coherence and the areal extremes, which matter for
the floods (rain falling at the same time over a catchment)?

**Method.** The dependence statistics of `vu.dependence_stats`, already cached by 5 (ERA5, raw, F1), 6 (ranked jitter),
7 (MBCn historical) and 7b (runs, first 34 years, every version): mean correlation of the daily series of all pairs of
domain pixels per distance class and its e-folding distance (pr and tasmax anomaly, DJF and JJA); extremes of the
areal mean pr over blocks of 3 × 3 and 6 × 6 cells (about 225 and 450 km: median over the blocks of the mean annual
Rx1day and Rx5day, and the areal reduction factor ARF = areal Rx1day / mean Rx1day of the pixels). Historical against
ERA5, and the changes (warming, AMOC weakening) of every version against raw. Nothing is recomputed; writes only
`outputs/` next to the notebook.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
CAL_PERIOD = ("1981", "2014")      # as F1
REGION = bu.gcm_ba_region(cfg, 1.0)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}" + ("_europe" if REGION else "")
RANK_TAG = f"{F1_TAG}_rankjit"                      # products of 6_test_jitter
TEST_TAG = f"{F1_TAG}_mbcn"
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUNS = ["gcm_historical"] + GCM_RUNS
TEMP_SNOW = 1.0                    # degC, mean daily temperature below which pr falls as snow (LISFLOOD TempSnow)
TEMP_MELT = 1.0                    # degC, threshold of the positive degree-days (LISFLOOD TempMelt, 0 or 1 in its docs)
COLD = [11, 12, 1, 2, 3, 4]        # cold season of the snow statistics
N_ITER = 20                        # MBCn iterations (Cannon 2018)
NQ = 50                            # quantiles of the npdf transform and of the final QDM
WINDOW = 19                        # windows of 5 days x 19 = 95 days (about the 3 months of F1)
SEED = 20261109
LOC_CHUNK = 400                    # domain pixels per dask block
OVERWRITE = False
MOUNTAINS = {"Alps": {"lat": (43.5, 48.5), "lon": (5.0, 16.5)}, "Scandinavia": {"lat": (58.0, 71.0), "lon": (4.0, 20.0)}}

DOMAIN = vu.init(cfg, "domain", borders_res="01M")        # LISFLOOD domain without the land outside Europe (as 5)
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")    # F1 products (notebook 3)
RANK = bu.gcm_ba_dir(cfg, "6_test_jitter", RANK_TAG)       # products and cache of 6_test_jitter (read only)
CACHE5 = bu.gcm_ba_dir(cfg, "5_dependence", "cache" + ("_europe" if REGION else ""))   # statistics of 5 (read only)
TEST7 = bu.gcm_ba_dir(cfg, "7_test_mbcn", TEST_TAG)       # 7: products and caches (read only)
PDIR, CACHE7 = TEST7 / "products", TEST7 / "cache"
CACHE = bu.gcm_ba_dir(cfg, "7b_test_mbcn_first34y", TEST_TAG) / "cache"   # 7b (read only): runs, first 34 years
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
KINDS = ["raw", "F1", "rank", "mbcn"]
period_of = lambda name: CAL_PERIOD if name in ("era5", "gcm_historical") else None
print(f"F1 {F1_TAG} | ranked jitter {RANK_TAG} | MBCn {N_ITER} iterations, {NQ} quantiles, 5 d x {WINDOW} windows | "
      f"snow below {TEMP_SNOW} degC | outputs in {OUT}")

## 1. Dependence statistics of every version (cached by 5, 6, 7 and 7b)

In [ ]:
def dep_path(name, k):
    tag = {"raw": "raw", "F1": F1_TAG, "rank": RANK_TAG, "mbcn": TEST_TAG}[k]
    if name in ("era5", "gcm_historical"):                      # whole period: caches of 5, 6, 7
        p = {"raw": CACHE5, "F1": CACHE5, "rank": RANK / "cache", "mbcn": CACHE7}[k] / f"{name}_{tag}_dependence.pkl"
        return next((q for q in (p, CACHE7 / p.name, CACHE / p.name) if q.exists()), p)
    return CACHE / f"{name}_{tag}_first34y_dependence.pkl"       # runs: first 34 years (7b)


DEP = {}
for name, kinds in [("era5", ["raw"])] + [(n, KINDS) for n in RUNS]:
    for k in kinds:
        key = "ERA5" if name == "era5" else f"{LABEL[name]} {k}"
        p = dep_path(name, k)
        assert p.exists(), f"missing {p} (run 5, 6, 7 and 7b first)"
        DEP[key] = pd.read_pickle(p)
COLS = list(DEP["ERA5"]["spatial"].columns)
ef = lambda d, c: vu.efold_km(d["spatial"].index.values, d["spatial"][c].values)
areal = lambda d, k, st: float(np.median(d["areal"][k][st]))
print(len(DEP), "datasets")

## 2. Historical 1981–2014 against ERA5

In [ ]:
VERS = ["ERA5"] + [f"historical {k}" for k in KINDS]
SP = pd.DataFrame({v.replace("historical ", ""): {f"{c[0]}, {c[1]}": ef(DEP[v], c) for c in COLS} for v in VERS})
SP.index.name = "e-folding distance (km)"
rows = []
for k in vu.DEP_BLOCKS[1:]:
    for st in ("Rx1day", "Rx5day", "ARF"):
        rows.append({"statistic": f"{st}, {k} x {k} cells", **{v.replace("historical ", ""): areal(DEP[v], k, st) for v in VERS}})
AR = pd.DataFrame(rows).set_index("statistic")
SP.to_csv(OUT / f"spatial_efold_historical_{TEST_TAG}.csv"); AR.to_csv(OUT / f"areal_extremes_historical_{TEST_TAG}.csv")
with pd.option_context("display.width", 250):
    print("(1) e-folding distance of the mean correlation between pixels (km)"); display(SP.round(0))
    print("(2) areal extremes, median over the blocks (mm/day, mm / 5 days, ARF)"); display(AR.round(3))
fig, axes = plt.subplots(1, len(COLS), figsize=(5 * len(COLS), 3.6))
for ax, c in zip(axes, COLS):
    for v, st in zip(VERS, ("k-", "C0--", "C3-", "C2-", "C1-")):
        ax.plot(DEP[v]["spatial"].index, DEP[v]["spatial"][c], st, label=v.replace("historical ", ""))
    ax.axhline(np.exp(-1), color="0.6", lw=0.8); ax.set_ylim(-0.1, 1)
    ax.set_title(f"{c[0]}, {c[1]}", fontsize=9); ax.set_xlabel("distance (km)")
axes[0].set_ylabel("mean correlation"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "spatial_correlation_historical.png", dpi=90); plt.show()

## 3. Changes (runs: first 34 years): e-folding distances and areal extremes, every version against raw

In [ ]:
rows = []
for a_, b_, lab in PAIRS:
    ka, kb = LABEL[a_], LABEL[b_]
    for c in COLS:
        rows.append({"pair": lab, "statistic": f"e-folding {c[0]}, {c[1]} (km)",
                     **{f"change {k}": ef(DEP[f"{ka} {k}"], c) - ef(DEP[f"{kb} {k}"], c) for k in KINDS}})
    for k in vu.DEP_BLOCKS[1:]:
        for st in ("Rx1day", "Rx5day", "ARF"):
            f = (lambda x1, x0: 100 * (x1 / x0 - 1)) if st != "ARF" else (lambda x1, x0: x1 - x0)
            rows.append({"pair": lab, "statistic": f"areal {st}, {k} x {k} cells" + (" (%)" if st != "ARF" else ""),
                         **{f"change {t}": f(areal(DEP[f"{ka} {t}"], k, st), areal(DEP[f"{kb} {t}"], k, st)) for t in KINDS}})
CH = pd.DataFrame(rows).set_index(["pair", "statistic"])
CH.to_csv(OUT / f"spatial_areal_changes_{TEST_TAG}.csv")
with pd.option_context("display.width", 250, "display.max_rows", 200):
    display(CH.round(3))